# 04_train_model — DataPulse MLOps no Databricks Free Edition

Notebook completo para treinar um modelo ou executar todos os experimentos, registrar parâmetros, métricas, assinatura e artefatos no MLflow.

**Configuração esperada:** os dados estão em `datapulse.gold.failure_training` (Unity Catalog/Delta) ou em um CSV acessível ao driver. Ajuste `DATA_SOURCE` e `TARGET_COLUMN` na célula de configuração. O target padrão é `falha_24h`.

> Para tabelas grandes, evite converter todo o dataset para Pandas. Este notebook usa Pandas porque os estimadores selecionados são do Scikit-learn/XGBoost e o exemplo é voltado a conjuntos de dados de tamanho gerenciável.

In [0]:
#Variaveis de ambiente

# 2. Configuração do notebook
# Opção recomendada no Databricks: tabela Gold do Unity Catalog.
DATA_SOURCE = "datapulse.gold.sensor_events"
try:
    CATALOG = dbutils.widgets.get("catalog")
except:
    catalog = "dev"
SCHEMA = "falha"

In [0]:
# 1. Imports
import os
import warnings
from typing import Optional, Dict, Any

from sklearn.model_selection import GridSearchCV, StratifiedKFold

import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from mlflow.models import infer_signature

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from typing import Tuple

import pandas as pd
from pyspark.sql import SparkSession

warnings.filterwarnings("default")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    print("XGBoost não está instalado; o modelo será omitido.")


XGBoost não está instalado; o modelo será omitido.


In [0]:


# Alternativa CSV: informe um caminho acessível ao driver do cluster,
# por exemplo "/Volumes/main/default/industrial_data/industrial_data_serving.csv".
# Não use um caminho local do seu computador Windows.
DATA_SOURCE_TYPE = "table"  # "table" ou "csv"


TEST_SIZE = 0.20
RANDOM_STATE = 42

TARGET_COLUMN = "big_falha_24h"


EXPERIMENT_NAME = "/Shared/DataPulse/FailurePrediction"

# Para registrar versões no Unity Catalog, defina por exemplo:
# REGISTERED_MODEL_NAME = "datapulse.ml.failure_predictor"
# O registro é opcional e requer permissões no Unity Catalog.
REGISTERED_MODEL_NAME = None

mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow experiment:", EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())

# Teste de hiperparametros

PARAM_GRID = {
    "GaussianNB": {
        "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]
    },

    "KNeighborsClassifier": {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    },

    "SVC": {
        "C": [0.1, 1, 10, 100],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale", "auto"],
    },

    "GradientBoostingClassifier": {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
    },

    "RandomForestClassifier": {
        "n_estimators": [100, 200, 300],
        "max_depth": [5, 10, 15, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    },

    "LogisticRegression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs", "liblinear"],
        "max_iter": [5000, 10000],
    },

    "DecisionTreeClassifier": {
        "max_depth": [3, 5, 10, 15, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "criterion": ["gini", "entropy"],
    },
}

if XGBOOST_AVAILABLE:
    PARAM_GRID["XGBClassifier"] = {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    }


MLflow experiment: /Shared/DataPulse/FailurePrediction
Tracking URI: databricks


In [0]:
# 5. Registry centralizado dos estimadores
MODEL_REGISTRY = {
    "GaussianNB": GaussianNB,
    "KNeighborsClassifier": KNeighborsClassifier,
    "SVC": SVC,
    "GradientBoostingClassifier": GradientBoostingClassifier,
    "RandomForestClassifier": RandomForestClassifier,
    "LogisticRegression": LogisticRegression,
    "DecisionTreeClassifier": DecisionTreeClassifier,
}

if XGBOOST_AVAILABLE:
    MODEL_REGISTRY["XGBClassifier"] = XGBClassifier

print("Modelos disponíveis:", list(MODEL_REGISTRY.keys()))


Modelos disponíveis: ['GaussianNB', 'KNeighborsClassifier', 'SVC', 'GradientBoostingClassifier', 'RandomForestClassifier', 'LogisticRegression', 'DecisionTreeClassifier']


In [0]:

# 6. Treinamento, otimização, avaliação e logging no MLflow

def train_model(
    model,
    model_name: str,
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    registered_model_name: Optional[str] = REGISTERED_MODEL_NAME,
    param_grid: Optional[Dict[str, Any]] = None,
    cv: int = 5,
    scoring: str = "f1_weighted",
    n_jobs: int = -1,
) -> Dict[str, Any]:

    """
    Otimiza hiperparâmetros com GridSearchCV, treina o melhor
    estimador e registra parâmetros, métricas e artefatos no MLflow.
    """

    with mlflow.start_run(run_name=model_name) as run:

        mlflow.set_tags({
            "project": "DataPulse",
            "task": "industrial_failure_classification",
            "model_name": model_name,
            "target_column": TARGET_COLUMN,
            "optimization": "GridSearchCV",
        })

        # --------------------------------------------------
        # 1. Configuração da validação cruzada
        # --------------------------------------------------

        cv_strategy = StratifiedKFold(
            n_splits=cv,
            shuffle=True,
            random_state=RANDOM_STATE,
        )

        if param_grid is None:
            param_grid = PARAM_GRID.get(type(model).__name__, {})

        # --------------------------------------------------
        # 2. Otimização dos hiperparâmetros
        # --------------------------------------------------

        print(f"\nStarting GridSearchCV: {model_name}")

        if param_grid:

            grid_search = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                scoring=scoring,
                cv=cv_strategy,
                n_jobs=n_jobs,
                verbose=1,
                refit=True,
                error_score="raise",
            )

            grid_search.fit(X_train, y_train)

            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_
            best_score = float(grid_search.best_score_)

            print("Best parameters:", best_params)
            print("Best CV score:", best_score)

            # Registrar hiperparâmetros encontrados
            mlflow.log_params({
                f"best_{key}": value
                for key, value in best_params.items()
            })

            mlflow.log_metric("best_cv_score", best_score)

            # Salvar resultados completos da busca
            cv_results = pd.DataFrame(grid_search.cv_results_)

            cv_results_path = "/tmp/grid_search_results.csv"
            cv_results.to_csv(cv_results_path, index=False)

            mlflow.log_artifact(
                cv_results_path,
                artifact_path="optimization",
            )

        else:
            print("No parameter grid found. Using default parameters.")

            best_model = model
            best_model.fit(X_train, y_train)

            best_params = best_model.get_params()
            best_score = None

        # --------------------------------------------------
        # 3. Registrar parâmetros do modelo
        # --------------------------------------------------

        mlflow.log_params({
            f"final_{key}": str(value)
            for key, value in best_model.get_params().items()
            if value is not None
        })

        mlflow.log_params({
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            "feature_count": X_train.shape[1],
            "test_size": TEST_SIZE,
            "random_state": RANDOM_STATE,
            "cv_folds": cv,
            "optimization_scoring": scoring,
        })

        # --------------------------------------------------
        # 4. Avaliação no conjunto de teste
        # --------------------------------------------------

        y_pred = best_model.predict(X_test)

        average = (
            "binary"
            if y_test.nunique() == 2
            else "weighted"
        )

        metrics = {
            "accuracy": float(
                accuracy_score(y_test, y_pred)
            ),
            "precision": float(
                precision_score(
                    y_test,
                    y_pred,
                    average=average,
                    zero_division=0,
                )
            ),
            "recall": float(
                recall_score(
                    y_test,
                    y_pred,
                    average=average,
                    zero_division=0,
                )
            ),
            "f1": float(
                f1_score(
                    y_test,
                    y_pred,
                    average=average,
                    zero_division=0,
                )
            ),
        }

        # ROC AUC
        try:

            if y_test.nunique() == 2:

                if hasattr(best_model, "predict_proba"):

                    proba = best_model.predict_proba(X_test)

                    positive_label = (
                        1 if 1 in list(best_model.classes_)
                        else best_model.classes_[-1]
                    )

                    positive_idx = list(
                        best_model.classes_
                    ).index(positive_label)

                    y_binary = (
                        y_test == positive_label
                    ).astype(int)

                    metrics["roc_auc"] = float(
                        roc_auc_score(
                            y_binary,
                            proba[:, positive_idx],
                        )
                    )

                elif hasattr(best_model, "decision_function"):

                    scores = best_model.decision_function(X_test)

                    metrics["roc_auc"] = float(
                        roc_auc_score(y_test, scores)
                    )

        except Exception as exc:
            print(f"Warning: ROC AUC unavailable: {exc}")

        mlflow.log_metrics(metrics)

        # --------------------------------------------------
        # 5. Registrar modelo no MLflow
        # --------------------------------------------------

        signature = infer_signature(X_test, y_pred)

        input_example = X_test.head(
            min(5, len(X_test))
        )

        model_info = mlflow.sklearn.log_model(
            sk_model=best_model,
            name="model",
            signature=signature,
            input_example=input_example,
        )

        # --------------------------------------------------
        # 6. Relatório de classificação
        # --------------------------------------------------

        report = classification_report(
            y_test,
            y_pred,
            zero_division=0,
        )

        report_path = "/tmp/classification_report.txt"

        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report)

        mlflow.log_artifact(
            report_path,
            artifact_path="evaluation",
        )

        # --------------------------------------------------
        # 7. Matriz de confusão
        # --------------------------------------------------

        labels = sorted(
            pd.Series(y_test).unique().tolist()
        )

        cm = confusion_matrix(
            y_test,
            y_pred,
            labels=labels,
        )

        cm_df = pd.DataFrame(
            cm,
            index=[f"actual_{label}" for label in labels],
            columns=[f"predicted_{label}" for label in labels],
        )

        cm_path = "/tmp/confusion_matrix.csv"

        cm_df.to_csv(cm_path, index=True)

        mlflow.log_artifact(
            cm_path,
            artifact_path="evaluation",
        )

        # --------------------------------------------------
        # 8. Model Registry
        # --------------------------------------------------

        registry_version = None

        if registered_model_name:

            registered = mlflow.register_model(
                model_uri=model_info.model_uri,
                name=registered_model_name,
            )

            registry_version = str(registered.version)

        # --------------------------------------------------
        # 9. Resultado
        # --------------------------------------------------

        result = {
            "model_name": model_name,
            "model_type": type(best_model).__name__,
            "run_id": run.info.run_id,
            "model_uri": model_info.model_uri,
            "registered_model_version": registry_version,
            "best_params": best_params,
            "best_cv_score": best_score,
            **metrics,
        }

        print(f"Run ID: {run.info.run_id}")
        print("Best parameters:", best_params)
        print("Metrics:", metrics)
        print("Classification report:\n", report)

        return result


In [0]:
def get_data(
    catalog: str,
    schema: str,
    target_col: str = "big_falha_24h",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:

    spark = SparkSession.getActiveSession()

    if spark is None:
        raise RuntimeError("Nenhuma sessão Spark ativa encontrada.")

    # Carrega tabelas Delta
    train_df = spark.table(f"{catalog}.{schema}.train_set")
    test_df = spark.table(f"{catalog}.{schema}.test_set")

    # Converte para Pandas
    train_pd = train_df.toPandas()
    test_pd = test_df.toPandas()

    # Separa features e target
    X_train = train_pd.drop(columns=[target_col])
    y_train = train_pd[target_col]

    X_test = test_pd.drop(columns=[target_col])
    y_test = test_pd[target_col]

    return X_train, X_test, y_train, y_test

In [0]:
# 7. Executar um ou vários modelos a partir de configurações
def run_experiment(
    models: Optional[Dict[str, Dict[str, Any]]] = None,
    source: str = DATA_SOURCE,
    source_type: str = DATA_SOURCE_TYPE,
    target: str = TARGET_COLUMN,
) -> pd.DataFrame:
    """Executa os modelos declarados no dicionário de configuração."""

    X_train, X_test, y_train, y_test = get_data(CATALOG, SCHEMA, target_col=TARGET_COLUMN  )

    if models is None:
        models = {
            "GaussianNB_v1": {
                "model": "GaussianNB",
                "parameters": {},
            }
        }

    results = []
    for model_name, model_config in models.items():
        model_type = model_config["model"]
        parameters = model_config.get("parameters", {})

        if model_type not in MODEL_REGISTRY:
            raise ValueError(
                f"Modelo '{model_type}' não está disponível. "
                f"Modelos disponíveis: {list(MODEL_REGISTRY.keys())}"
            )

        model_class = MODEL_REGISTRY[model_type]
        model = model_class(**parameters)

        print(f"\n{'=' * 60}")
        print(f"Training model: {model_name}")
        print(f"Model type: {model_type}")
        print(f"Parameters: {parameters}")
        print(f"{'=' * 60}")

        result = train_model(
            model=model,
            model_name=model_name,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
        )
        results.append(result)

    results_df = pd.DataFrame(results)
    if not results_df.empty and "f1" in results_df.columns:
        results_df = results_df.sort_values("f1", ascending=False).reset_index(drop=True)

    return results_df


In [0]:
# 8. Executar todos os experimentos
def run_allexperiments(
    source: str = DATA_SOURCE,
    source_type: str = DATA_SOURCE_TYPE,
    target: str = TARGET_COLUMN,
) -> Dict[str, Any]:
    """Executa a lista padrão de modelos usando o mesmo split e o mesmo pipeline MLflow."""
    print("Starting model training...")

    models = {
        "GaussianNB": {
            "model": "GaussianNB",
            "parameters": {},
        },
        "KNN": {
            "model": "KNeighborsClassifier",
            "parameters": {"n_neighbors": 5, "weights": "distance"},
        },
        "SVM": {
            "model": "SVC",
            "parameters": {
                "C": 1.0, "kernel": "rbf",
                "probability": True, "random_state": 42,
            },
        },
        "GradientBoosting": {
            "model": "GradientBoostingClassifier",
            "parameters": {
                "n_estimators": 200, "learning_rate": 0.05,
                "max_depth": 5, "random_state": 42,
            },
        },
        "RandomForest": {
            "model": "RandomForestClassifier",
            "parameters": {
                "n_estimators": 300, "max_depth": 10,
                "min_samples_split": 10, "random_state": 42,
                "n_jobs": -1,
            },
        },
        "LogisticRegression": {
            "model": "LogisticRegression",
            "parameters": {
                "max_iter": 10000, "C": 1.0, "random_state": 42,
            },
        },
        "DecisionTree": {
            "model": "DecisionTreeClassifier",
            "parameters": {
                "max_depth": 5, "min_samples_split": 10,
                "random_state": 42,
            },
        },
    }

    if XGBOOST_AVAILABLE:
        models["XGBoost"] = {
            "model": "XGBClassifier",
            "parameters": {
                "n_estimators": 300, "learning_rate": 0.1,
                "max_depth": 6, "random_state": 42,
                "n_jobs": -1, "eval_metric": "logloss",
            },
        }

    results_df = run_experiment(
        models=models,
        source=source,
        source_type=source_type,
        target=target,
    )

    print("All experiments executed successfully.")
    display(results_df)

    return {
        "status": "success",
        "total_experiments": len(results_df),
        "results": results_df,
    }


## 9. Executar o treinamento

Escolha **uma** das células abaixo para executar.

In [0]:
# Opção C — executar todos os modelos
# Descomente e execute esta célula no lugar da Opção A.
#
all_results = run_allexperiments()
print(all_results["status"])
print("Total de experimentos:", all_results["total_experiments"])


Starting model training...

Training model: GaussianNB
Model type: GaussianNB
Parameters: {}

Starting GridSearchCV: GaussianNB
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best parameters: {'var_smoothing': 1e-06}
Best CV score: 0.9067477159165396


🔗 View Logged Model at: https://dbc-0395191b-4103.cloud.databricks.com/ml/experiments/3079201492020369/models/m-f7822146f2c749f8ad1f3e6ef8684914?o=7474646727692125


Run ID: 7471cd722f8d44c2ba786ed59a510e3f
Best parameters: {'var_smoothing': 1e-06}
Metrics: {'accuracy': 0.9074074074074074, 'precision': 0.18333333333333332, 'recall': 0.2018348623853211, 'f1': 0.19213973799126638, 'roc_auc': 0.7575970976343}
Classification report:
               precision    recall  f1-score   support

           0       0.95      0.95      0.95      1889
           1       0.18      0.20      0.19       109

    accuracy                           0.91      1998
   macro avg       0.57      0.57      0.57      1998
weighted avg       0.91      0.91      0.91      1998


Training model: KNN
Model type: KNeighborsClassifier
Parameters: {'n_neighbors': 5, 'weights': 'distance'}

Starting GridSearchCV: KNN
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters: {'n_neighbors': 5, 'p': 1, 'weights': 'uniform'}
Best CV score: 0.9200118126550316


🔗 View Logged Model at: https://dbc-0395191b-4103.cloud.databricks.com/ml/experiments/3079201492020369/models/m-93482cdf20d9445da4b1096eb1058760?o=7474646727692125


Run ID: 8829056fc564448d86c28209dd5b064b
Best parameters: {'n_neighbors': 5, 'p': 1, 'weights': 'uniform'}
Metrics: {'accuracy': 0.9434434434434434, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'roc_auc': 0.5231106211237438}
Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97      1889
           1       0.00      0.00      0.00       109

    accuracy                           0.94      1998
   macro avg       0.47      0.50      0.49      1998
weighted avg       0.89      0.94      0.92      1998


Training model: SVM
Model type: SVC
Parameters: {'C': 1.0, 'kernel': 'rbf', 'probability': True, 'random_state': 42}

Starting GridSearchCV: SVM
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best CV score: 0.9190770992873587


🔗 View Logged Model at: https://dbc-0395191b-4103.cloud.databricks.com/ml/experiments/3079201492020369/models/m-414f136e68f14d398813998ac84b11b3?o=7474646727692125


... (7 outputs left out)

🔗 View Logged Model at: https://dbc-0395191b-4103.cloud.databricks.com/ml/experiments/3079201492020369/models/m-b1ba673754aa4dbab378375a70de0104?o=7474646727692125


Run ID: 82e2cb638eb343579e716b377394d160
Best parameters: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 2}
Metrics: {'accuracy': 0.943943943943944, 'precision': 0.2, 'recall': 0.009174311926605505, 'f1': 0.017543859649122806, 'roc_auc': 0.7024030966338192}
Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97      1889
           1       0.20      0.01      0.02       109

    accuracy                           0.94      1998
   macro avg       0.57      0.50      0.49      1998
weighted avg       0.91      0.94      0.92      1998

All experiments executed successfully.


model_name,model_type,run_id,model_uri,registered_model_version,best_params,best_cv_score,accuracy,precision,recall,f1,roc_auc
GaussianNB,GaussianNB,7471cd722f8d44c2ba786ed59a510e3f,models:/m-f7822146f2c749f8ad1f3e6ef8684914,null,"List(null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1.0E-6, null)",0.9067477159165396,0.9074074074074074,0.18333333333333332,0.2018348623853211,0.19213973799126638,0.7575970976343
DecisionTree,DecisionTreeClassifier,82e2cb638eb343579e716b377394d160,models:/m-b1ba673754aa4dbab378375a70de0104,null,"List(null, gini, null, null, null, 3, null, null, 4, 2, null, null, null, null, null, null, null)",0.9197454056452037,0.943943943943944,0.2,0.009174311926605505,0.017543859649122806,0.7024030966338192
GradientBoosting,GradientBoostingClassifier,13e2d65fc01c4f7a875f338f56f76abb,models:/m-1a6c9c9336554c81bae8ee33fa77d97a,null,"List(null, null, null, null, 0.05, 5, null, null, null, null, 100, null, null, null, 1.0, null, null)",0.920916840499651,0.9434434434434434,0.16666666666666666,0.009174311926605505,0.017391304347826087,0.7157323179586307
KNN,KNeighborsClassifier,8829056fc564448d86c28209dd5b064b,models:/m-93482cdf20d9445da4b1096eb1058760,null,"List(null, null, null, null, null, null, null, null, null, null, null, 5, 1, null, null, null, uniform)",0.9200118126550316,0.9434434434434434,0.0,0.0,0.0,0.5231106211237438
SVM,SVC,6431252e47c44df5ada9a9fc4932440d,models:/m-414f136e68f14d398813998ac84b11b3,null,"List(0.1, null, scale, linear, null, null, null, null, null, null, null, null, null, null, null, null, null)",0.9190770992873587,0.9454454454454454,0.0,0.0,0.0,0.7389473581964148
RandomForest,RandomForestClassifier,89c84442acb149ae820c7bb376313e09,models:/m-76174095c86045019ac61375b76a412f,null,"List(null, null, null, null, null, 15, sqrt, null, 1, 10, 200, null, null, null, null, null, null)",0.9193837116756238,0.9454454454454454,0.0,0.0,0.0,0.702944619015935
LogisticRegression,LogisticRegression,18e446f7afe24574902ef463026bc916,models:/m-f28fcc4f37b446f7afe7530117c8ed75,null,"List(0.01, null, null, null, null, null, null, 5000, null, null, null, null, null, lbfgs, null, null, null)",0.9190770992873587,0.9454454454454454,0.0,0.0,0.0,0.7680244389293884


success
Total de experimentos: 7


## 10. Conferir execuções no MLflow

Abra o experimento `/Shared/DataPulse/FailurePrediction` na interface do MLflow do Databricks. Cada modelo terá sua própria execução com parâmetros, métricas e artefatos.

### Observações importantes
- Se estiver usando CSV, altere `DATA_SOURCE_TYPE = "csv"` e `DATA_SOURCE` para um caminho acessível ao cluster, preferencialmente um Unity Catalog Volume.
- A tabela deve conter `falha_24h` e somente features utilizáveis pelo modelo. Identificadores, timestamps brutos e colunas de vazamento devem ser removidos ou transformados antes do treinamento.
- O notebook faz imputação de valores numéricos usando a mediana do treino. Para produção, é melhor colocar a imputação dentro de um `Pipeline` do Scikit-learn e registrar o pipeline inteiro.
- O uso de `toPandas()` exige que o dataset caiba na memória do driver.
- O registro opcional no Unity Catalog exige permissões e configuração compatíveis. Mantenha `REGISTERED_MODEL_NAME = None` até confirmar essa configuração.
- Métricas em datasets desbalanceados devem priorizar recall, precision, F1 e matriz de confusão, não apenas accuracy.
